# Colab training — UAV Multi-Priority Resource Allocation (SAC + Ensemble Critic)

Run this notebook on **Google Colab** (free tier, GPU optional). It:
1. Mounts Google Drive for checkpoints/results
2. Clones the repo and installs the package
3. Trains SAC / REDQ-SAC with checkpoints every 10k steps and `--resume`
4. Copies `results/<run_name>/` back to Drive

See `plan.md` Sections 2.2 and 6 (Phases 3–9).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = "git@github.com:tra134/Multi-Priority-Resource-Allocation-in-UAV-Networks-research.git"
RUN_NAME = "sac_seed0_N10"

import os
os.chdir("/content")
if not os.path.exists("uav-drl-llm"):
    !git clone {REPO_URL}
os.chdir("/content/uav-drl-llm")
!pip install -e .

In [ ]:
import os

ckpt_dir = "/content/drive/MyDrive/uavrl_ckpt/" + RUN_NAME
os.makedirs(ckpt_dir, exist_ok=True)

# Phase 3 (Pure SAC): --agent sac (N_critics=2 from config gives twin critics)
# Phase 4 (REDQ-SAC): set configs/default.yaml sac.N_critics=10 before running
!python scripts/train.py \
    --config configs/default.yaml \
    --agent sac \
    --seed 0 \
    --steps 300000 \
    --ckpt-dir {ckpt_dir} \
    --resume \
    --eval-every 20000 \
    --eval-episodes 20 \
    --tb-dir /content/drive/MyDrive/uavrl_tb/{RUN_NAME}

In [ ]:
# Copy finished results back to Drive so they survive session disconnects
!cp -r /content/uav-drl-llm/results /content/drive/MyDrive/uavrl_results/{RUN_NAME} 2>/dev/null || true
!cp -r /content/uav-drl-llm/results/ckpt /content/drive/MyDrive/uavrl_ckpt/ 2>/dev/null || true
print("Done. Results copied to Drive.")